# Kraken vs TrOCR benchmark na pelnych stronach EHRI

Porownanie trzech pipeline'ow na pelnych polskich stronach maszynopisu z EHRI:
1. **OpenCV + TrOCR run5** (`trocr-pl-mixed-v3`) - najlepszy model na druk
2. **OpenCV + TrOCR run6** (`trocr-pl-mixed-aug-v1`) - najlepszy model na maszynopis
3. **Kraken end-to-end** (`polish_nfd_9313.mlmodel`) - model EHRI (93,1% accuracy)

Cel: sprawdzic czy Kraken (segmentacja baseline + .mlmodel) rozwiazuje problem
pelnych stron maszynopisu, gdzie OpenCV+TrOCR zawodzi przez slaba segmentacje.

Metryka: CER/WER na poziomie linii (GT z ALTO XML).

In [ ]:
import subprocess, sys, os
from pathlib import Path
CODE_REVISION = 'c1eb95931c5c4e29bc5d844437cf212848b1a3b2'
BASE_MODEL = 'PiotrSty/trocr-pl-base'
RUN5_MODEL = 'PiotrSty/trocr-pl-mixed-v3'
RUN6_MODEL = 'PiotrSty/trocr-pl-mixed-aug-v1'
EHRI_REVISION = '3003e8614b74a351e7d94aba4f1348368815fb70'
EHRI_DATASET_REPO = 'PiotrSty/ehri-dataset'
KRAKEN_MODEL_FILE = 'models/polish_nfd_9313.mlmodel'
repo = Path('/kaggle/working/OCR_engine')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/PiotrStyla/OCR_engine.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin',CODE_REVISION],check=True)
subprocess.run(['git','-C',str(repo),'checkout','--detach',CODE_REVISION],check=True)
os.chdir(repo)
sys.path.insert(0,str(repo))
subprocess.run([sys.executable,'-m','pip','install','kraken>=5.0','jiwer','pillow','opencv-python-headless'],check=True)
subprocess.run([sys.executable,'-m','pip','uninstall','-y','huggingface_hub'],check=True)
subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','huggingface_hub==0.36.2'],check=True)
subprocess.run([sys.executable,'-m','pip','install','transformers==4.57.6','peft==0.19.1','accelerate'],check=True)
for _mod in list(sys.modules):
    if _mod == 'huggingface_hub' or _mod.startswith('huggingface_hub.') or _mod == 'transformers' or _mod.startswith('transformers.'):
        del sys.modules[_mod]
from huggingface_hub.utils import HfFolder  # noqa: F401
from transformers import VisionEncoderDecoderModel  # noqa: F401
import kraken
from importlib.metadata import version as _pkg_version
print('IMPORT_OK', 'kraken', _pkg_version('kraken'))

In [ ]:
import torch, json, tarfile, shutil
from pathlib import Path
from huggingface_hub import hf_hub_download, list_repo_files
assert torch.cuda.is_available(), 'GPU required; select Kaggle GPU T4.'
print('GPU:', torch.cuda.get_device_name(0))

ehri_archive = hf_hub_download('PiotrSty/ehri-pl-lines','ehri-pl-lines-v1.tar.gz',repo_type='dataset',revision=EHRI_REVISION)
ehri_root = Path('/kaggle/working/ehri-pl-lines-v1'); ehri_root.mkdir(parents=True, exist_ok=True)
with tarfile.open(ehri_archive,'r:gz') as b: b.extractall(ehri_root, filter='data')

kraken_model_path = hf_hub_download(EHRI_DATASET_REPO, KRAKEN_MODEL_FILE, repo_type='dataset')
print('Kraken model:', kraken_model_path)

# Pobierz WSZYSTKIE polskie strony .tif + .xml (15 stron, 468 linii)
ehri_pages_dir = Path('/kaggle/working/ehri-pages')
ehri_pages_dir.mkdir(parents=True, exist_ok=True)
ehri_xml_dir = Path('/kaggle/working/ehri-xml')
ehri_xml_dir.mkdir(parents=True, exist_ok=True)
all_files = list_repo_files(EHRI_DATASET_REPO, repo_type='dataset')
polish_files = [f for f in all_files if 'polish' in f and (f.endswith('.tif') or f.endswith('.xml'))]
for f in polish_files:
    path = hf_hub_download(EHRI_DATASET_REPO, f, repo_type='dataset')
    if f.endswith('.tif'):
        shutil.copy(path, ehri_pages_dir / Path(f).name)
    elif f.endswith('.xml'):
        shutil.copy(path, ehri_xml_dir / Path(f).name)

print('EHRI test lines:', len(list((ehri_root/'test').glob('*.png'))))
print('Full pages:', len(list(ehri_pages_dir.glob('*.tif'))))
print('ALTO XML:', len(list(ehri_xml_dir.glob('*.xml'))))

In [ ]:
# Benchmark 1: Line-level CER/WER - TrOCR models (run5 and run2-base; run6 not published on HF)
import sys, subprocess
from pathlib import Path
ehri_root = Path('/kaggle/working/ehri-pl-lines-v1')
real_lines = '/kaggle/working/OCR_engine/benchmarks/real-lines-v1/pairs'
for name, model in [('run5', RUN5_MODEL), ('run2-base', BASE_MODEL)]:
    for split, data in [('ehri-test', f'{ehri_root}/test'), ('real-lines-v1', real_lines)]:
        print(f'=== {name} on {split} ===', flush=True)
        subprocess.run([sys.executable,'-m','training.evaluate','--data',data,
            '--model',model,'--device','cuda','--batch-size','16'], check=True)

In [ ]:
# Line-level Kraken benchmark is SKIPPED — nienaturalny przypadek.
# Kraken jest trenowany na pełnych stronach z własną segmentacją (baseline detection).
# Pojedyncze cropy z sztucznym baseline dają CER 52% vs 7% raportowane na pełnych stronach.
# Full-page benchmark (komórka 5) jest właściwym testem dla Kraken.
print('SKIPPED: Kraken line-level (nienaturalny przypadek). Patrz komórka 5 — full-page benchmark.')
print()
print('Model diagnostic:')
import kraken.lib.models as models
recognizer = models.load_any(kraken_model_path, device='cuda')
nn = recognizer.nn if hasattr(recognizer, 'nn') else recognizer
print(f'  seg_type: {getattr(nn, "seg_type", "?")}')
print(f'  one_channel_mode: {getattr(nn, "one_channel_mode", "?")}')
batch, channels, height, width = nn.input
print(f'  input: batch={batch} channels={channels} height={height} width={width}')
print()
print('EHRI raportuje 93.1% accuracy (CER ~7%) na pełnych stronach z własną segmentacją.')
print('Line-level z sztucznym baseline daje CER 52% — nienaturalny przypadek użycia.')

In [ ]:
# Benchmark 4: Kraken + GT segmentacja + binarization (nlbin)
# Test czy binarization poprawia CER z 11.67% do ~7% (raport EHRI).
import jiwer, uuid, warnings, shutil
from pathlib import Path
from PIL import Image
import xml.etree.ElementTree as ET
import kraken.lib.models as models
from kraken.rpred import rpred
from kraken.containers import Segmentation, BaselineLine
from kraken import binarization
from huggingface_hub import hf_hub_download

warnings.filterwarnings('ignore')

ALTO_NS = {'alto': 'http://www.loc.gov/standards/alto/ns-v4#'}

def parse_alto_baselines(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []
    for tl in root.findall('.//alto:TextLine', ALTO_NS):
        baseline_str = tl.get('BASELINE', '')
        if not baseline_str:
            continue
        nums = [int(x) for x in baseline_str.split()]
        baseline = list(zip(nums[::2], nums[1::2]))
        polygon = tl.find('.//alto:Polygon', ALTO_NS)
        boundary = []
        if polygon is not None:
            pts_str = polygon.get('POINTS', '')
            nums = [int(x) for x in pts_str.split()]
            boundary = list(zip(nums[::2], nums[1::2]))
        text = ''
        for s in tl.findall('alto:String', ALTO_NS):
            text += s.get('CONTENT', '')
        lines.append((baseline, boundary, text))
    return lines

recognizer = models.load_any(kraken_model_path, device='cuda')

ehri_pages_dir = Path('/kaggle/working/ehri-pages')
ehri_xml_dir = Path('/kaggle/working/ehri-xml')
ehri_xml_dir.mkdir(parents=True, exist_ok=True)
# Pobierz ALTO XML (niezależnie od innych komórek)
for page_file in sorted(ehri_pages_dir.glob('*.tif')):
    xml_name = page_file.name.replace('.tif', '.xml')
    xml_path = ehri_xml_dir / xml_name
    if not xml_path.exists():
        downloaded = hf_hub_download(EHRI_DATASET_REPO, f'data/polish/{xml_name}', repo_type='dataset')
        shutil.copy(downloaded, xml_path)

all_refs, all_hyps = [], []
for page_file in sorted(ehri_pages_dir.glob('*.tif')):
    page_name = page_file.stem
    xml_path = ehri_xml_dir / f'{page_name}.xml'
    if not xml_path.exists():
        print(f'  {page_name}: no XML, skipping')
        continue
    gt_lines = parse_alto_baselines(xml_path)
    img = Image.open(page_file).convert('L')
    # Binarization (nlbin) — Kraken built-in
    img_bin = binarization.nlbin(img)
    kraken_lines = []
    for baseline, boundary, text in gt_lines:
        if not baseline or not boundary:
            continue
        if boundary[0] != boundary[-1]:
            boundary = boundary + [boundary[0]]
        line = BaselineLine(id=str(uuid.uuid4()), baseline=baseline, boundary=boundary)
        kraken_lines.append(line)
    seg = Segmentation(
        type='baselines', imagename=str(page_file), text_direction='horizontal-lr',
        script_detection=False, lines=kraken_lines,
    )
    pred = rpred(recognizer, img_bin, seg)
    hyp_lines = [record.prediction.strip() for record in pred]
    ref_lines = [t for _, _, t in gt_lines if t]
    all_refs.extend(ref_lines)
    all_hyps.extend(hyp_lines)
    print(f'  {page_name}: {len(ref_lines)} GT lines, {len(hyp_lines)} OCR lines')

cer = jiwer.cer(all_refs, all_hyps)
wer = jiwer.wer(all_refs, all_hyps)
print(f'\n=== Kraken + GT segmentacja + binarization (nlbin) ===')
print(f'linii:      {len(all_refs)}')
print(f'CER:        {cer:.4f}  ({cer*100:.2f}%)')
print(f'WER:        {wer:.4f}  ({wer*100:.2f}%)')

In [ ]:
# Benchmark 3: GT segmentacja z ALTO XML + Kraken recognizer (all 15 pages)
# Izoluje jakość recognizera od segmentacji.
import jiwer, uuid, warnings
from pathlib import Path
from PIL import Image
import xml.etree.ElementTree as ET
import kraken.lib.models as models
from kraken.rpred import rpred
from kraken.containers import Segmentation, BaselineLine

warnings.filterwarnings('ignore')

ALTO_NS = {'alto': 'http://www.loc.gov/standards/alto/ns-v4#'}

def parse_alto_baselines(xml_path):
    """Parsuje ALTO XML → lista (baseline, boundary, text). Używa namespace."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []
    for tl in root.findall('.//alto:TextLine', ALTO_NS):
        baseline_str = tl.get('BASELINE', '')
        if not baseline_str:
            continue
        nums = [int(x) for x in baseline_str.split()]
        baseline = list(zip(nums[::2], nums[1::2]))
        polygon = tl.find('.//alto:Polygon', ALTO_NS)
        boundary = []
        if polygon is not None:
            pts_str = polygon.get('POINTS', '')
            nums = [int(x) for x in pts_str.split()]
            boundary = list(zip(nums[::2], nums[1::2]))
        text = ''
        for s in tl.findall('alto:String', ALTO_NS):
            text += s.get('CONTENT', '')
        lines.append((baseline, boundary, text))
    return lines

recognizer = models.load_any(kraken_model_path, device='cuda')

ehri_pages_dir = Path('/kaggle/working/ehri-pages')
ehri_xml_dir = Path('/kaggle/working/ehri-xml')

all_refs, all_hyps = [], []
for page_file in sorted(ehri_pages_dir.glob('*.tif')):
    page_name = page_file.stem
    xml_path = ehri_xml_dir / f'{page_name}.xml'
    if not xml_path.exists():
        print(f'  {page_name}: no XML, skipping')
        continue
    gt_lines = parse_alto_baselines(xml_path)
    img = Image.open(page_file).convert('L')
    kraken_lines = []
    for baseline, boundary, text in gt_lines:
        if not baseline or not boundary:
            continue
        if boundary[0] != boundary[-1]:
            boundary = boundary + [boundary[0]]
        line = BaselineLine(id=str(uuid.uuid4()), baseline=baseline, boundary=boundary)
        kraken_lines.append(line)
    seg = Segmentation(
        type='baselines', imagename=str(page_file), text_direction='horizontal-lr',
        script_detection=False, lines=kraken_lines,
    )
    pred = rpred(recognizer, img, seg)
    hyp_lines = [record.prediction.strip() for record in pred]
    ref_lines = [t for _, _, t in gt_lines if t]
    all_refs.extend(ref_lines)
    all_hyps.extend(hyp_lines)
    print(f'  {page_name}: {len(ref_lines)} GT lines, {len(hyp_lines)} OCR lines')

cer = jiwer.cer(all_refs, all_hyps)
wer = jiwer.wer(all_refs, all_hyps)
print(f'\n=== Kraken + GT segmentacja (ALTO) — {len(all_refs)} lines, {len(list(ehri_pages_dir.glob("*.tif")))} pages ===')
print(f'CER:        {cer:.4f}  ({cer*100:.2f}%)')
print(f'WER:        {wer:.4f}  ({wer*100:.2f}%)')

In [ ]:
# Benchmark 2: Full-page OCR on EHRI test documents (all 15 pages, ALTO XML as GT)
import sys, json, uuid, warnings
from pathlib import Path
from PIL import Image
import xml.etree.ElementTree as ET
import jiwer

warnings.filterwarnings('ignore')
sys.path.insert(0, '/kaggle/working/OCR_engine')
from ocr.config import OcrConfig
from ocr.pipeline import OcrEngine

ehri_pages_dir = Path('/kaggle/working/ehri-pages')
ehri_xml_dir = Path('/kaggle/working/ehri-xml')

ALTO_NS = {'alto': 'http://www.loc.gov/standards/alto/ns-v4#'}

def load_page_gt_from_alto(xml_path):
    """Parsuje ALTO XML → lista tekstów linii (GT)."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []
    for tl in root.findall('.//alto:TextLine', ALTO_NS):
        text = ''
        for s in tl.findall('alto:String', ALTO_NS):
            text += s.get('CONTENT', '')
        if text:
            lines.append(text)
    return lines

for backend_name, backend_cfg in [
    ('OpenCV+TrOCR-run5', {'recognizer_backend': 'trocr', 'recognizer_pl': RUN5_MODEL, 'force_language': 'pl'}),
    ('Kraken-e2e', {'recognizer_backend': 'kraken', 'kraken_model': kraken_model_path, 'force_language': 'pl'}),
]:
    print(f'\n=== {backend_name} on full pages ===', flush=True)
    cfg = OcrConfig(**backend_cfg)
    cfg.device = 'cuda'
    cfg.kraken_binarize = False
    engine = OcrEngine(cfg)
    all_refs, all_hyps = [], []
    for page_file in sorted(ehri_pages_dir.glob('*.tif')):
        page_name = page_file.stem
        xml_path = ehri_xml_dir / f'{page_name}.xml'
        if not xml_path.exists():
            print(f'  {page_name}: no XML, skipping')
            continue
        gt_lines = load_page_gt_from_alto(xml_path)
        if not gt_lines:
            print(f'  {page_name}: no GT lines, skipping')
            continue
        result = engine.recognize(str(page_file))
        hyp_lines = [ln.text for ln in result.lines if ln.text]
        ref = '\n'.join(gt_lines)
        hyp = '\n'.join(hyp_lines)
        all_refs.append(ref)
        all_hyps.append(hyp)
        print(f'  {page_name}: {len(gt_lines)} GT lines, {len(hyp_lines)} OCR lines')
    if all_refs:
        cer = jiwer.cer(all_refs, all_hyps)
        wer = jiwer.wer(all_refs, all_hyps)
        print(f'  CER: {cer:.4f}  ({cer*100:.2f}%)')
        print(f'  WER: {wer:.4f}  ({wer*100:.2f}%)')
    engine.close()

In [ ]:
# Benchmark 5: Kraken e2e + deskew (OpenCV) — all 15 pages, ALTO XML as GT
import jiwer, warnings, sys
from pathlib import Path
from PIL import Image
import numpy as np
import xml.etree.ElementTree as ET

warnings.filterwarnings('ignore')
sys.path.insert(0, '/kaggle/working/OCR_engine')
from ocr.preprocess import deskew, load_image
from ocr.config import OcrConfig
from ocr.pipeline import OcrEngine

ehri_pages_dir = Path('/kaggle/working/ehri-pages')
ehri_xml_dir = Path('/kaggle/working/ehri-xml')

ALTO_NS = {'alto': 'http://www.loc.gov/standards/alto/ns-v4#'}

def load_page_gt_from_alto(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []
    for tl in root.findall('.//alto:TextLine', ALTO_NS):
        text = ''
        for s in tl.findall('alto:String', ALTO_NS):
            text += s.get('CONTENT', '')
        if text:
            lines.append(text)
    return lines

# Deskew + zapisz obrócone strony
deskewed_dir = Path('/kaggle/working/ehri-pages-deskewed')
deskewed_dir.mkdir(parents=True, exist_ok=True)
for page_file in sorted(ehri_pages_dir.glob('*.tif')):
    arr = load_image(str(page_file))
    arr_deskewed = deskew(arr)
    out_path = deskewed_dir / page_file.with_suffix('.png').name
    Image.fromarray(arr_deskewed).save(out_path)

# Kraken e2e na obróconych stronach
cfg = OcrConfig(recognizer_backend='kraken', kraken_model=kraken_model_path, force_language='pl')
cfg.device = 'cuda'
cfg.kraken_binarize = False
engine = OcrEngine(cfg)
all_refs, all_hyps = [], []
for page_file in sorted(deskewed_dir.glob('*.png')):
    page_name = page_file.stem
    xml_path = ehri_xml_dir / f'{page_name}.xml'
    if not xml_path.exists():
        continue
    gt_lines = load_page_gt_from_alto(xml_path)
    if not gt_lines:
        continue
    result = engine.recognize(str(page_file))
    hyp_lines = [ln.text for ln in result.lines if ln.text]
    ref = '\n'.join(gt_lines)
    hyp = '\n'.join(hyp_lines)
    all_refs.append(ref)
    all_hyps.append(hyp)
    print(f'  {page_name}: {len(gt_lines)} GT lines, {len(hyp_lines)} OCR lines')

if all_refs:
    cer = jiwer.cer(all_refs, all_hyps)
    wer = jiwer.wer(all_refs, all_hyps)
    print(f'\n=== Kraken e2e + deskew ===')
    print(f'CER: {cer:.4f}  ({cer*100:.2f}%)')
    print(f'WER: {wer:.4f}  ({wer*100:.2f}%)')
engine.close()